# LicitIA — Entrenamiento del Modelo Random Forest
**MS2 — ML Service**

Este notebook entrena el modelo de predicción con los datos reales del SECOP II.

**Requisitos previos:**
1. `python data/secop_loader.py` — descarga los datos
2. `python data/cleaner.py` — limpia el dataset
3. Ejecuta este notebook celda por celda

In [1]:
import pandas as pd
import numpy as np
import joblib
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score
from sklearn.preprocessing import LabelEncoder

print(' Librerías importadas')

 Librerías importadas


In [2]:
# Cargar dataset limpio
df = pd.read_csv('../data/processed/secop_limpio.csv', low_memory=False)
print(f'Dataset cargado: {len(df):,} filas')
print(f'Columnas: {list(df.columns)}')
print(f"\nDistribución objetivo:")
print(df['gano'].value_counts())

Dataset cargado: 28,557 filas
Columnas: ['id_contrato', 'nombre_entidad', 'nit_entidad', 'departamento', 'ciudad', 'tipo_de_contrato', 'modalidad_de_contratacion', 'valor_del_contrato', 'proveedor_adjudicado', 'fecha_de_firma', 'estado_contrato', 'cuantia', 'gano', 'log_cuantia']

Distribución objetivo:
gano
1    19038
0     9519
Name: count, dtype: int64


In [3]:
import numpy as np
import pandas as pd

# DEBUG: limpiar y ver columnas reales
df.columns = df.columns.str.strip().str.lower()

print(" Columnas del dataset:")
for col in df.columns:
    print(col)

# Preparar features — MISMO ORDEN que en feature_engineer.py
SECTORES_COMPETIDOS = ['OBRA', 'CONSULTORÍA', 'SUMINISTRO', 'INTERVENTORÍA', 'PRESTACIÓN DE SERVICIOS']
MODALIDADES_ABIERTAS = ['LICITACIÓN PÚBLICA', 'CONCURSO DE MÉRITOS', 'SELECCIÓN ABREVIADA']
ENTIDADES_ALTO_VOLUMEN = ['MINISTERIO', 'GOBERNACIÓN', 'ALCALDÍA', 'HOSPITAL', 'ESE', 'ICBF']

df['sector_u']    = df['tipo_de_contrato'].fillna('').str.upper()
df['modalidad_u'] = df['modalidad_de_contratacion'].fillna('').str.upper()
df['entidad_u']   = df['nombre_entidad'].fillna('').str.upper()

# USAR COLUMNA CORRECTA
COL_CUANTIA = 'valor_del_contrato'

# Construir la matriz X de features
X = pd.DataFrame({

    'log_cuantia':          np.log1p(df[COL_CUANTIA]),
    'cuantia_rango_medio':  ((df[COL_CUANTIA] >= 50e6) & (df[COL_CUANTIA] <= 800e6)).astype(int),
    'cuantia_muy_alta':     (df[COL_CUANTIA] > 2000e6).astype(int),
    'sector_competido':     df['sector_u'].apply(lambda s: int(any(x in s for x in SECTORES_COMPETIDOS))),
    'modalidad_abierta':    df['modalidad_u'].apply(lambda m: int(any(x in m for x in MODALIDADES_ABIERTAS))),
    'entidad_alto_volumen': df['entidad_u'].apply(lambda e: int(any(x in e for x in ENTIDADES_ALTO_VOLUMEN))),
    'tiene_nit':            df['nit_entidad'].notna().astype(int),
    'tiene_municipio':      df['ciudad'].notna().astype(int),  # 👈 también corregido aquí
})

y = df['gano']

print(f'Features construidos: {X.shape}')
print(X.head())

 Columnas del dataset:
id_contrato
nombre_entidad
nit_entidad
departamento
ciudad
tipo_de_contrato
modalidad_de_contratacion
valor_del_contrato
proveedor_adjudicado
fecha_de_firma
estado_contrato
cuantia
gano
log_cuantia
Features construidos: (28557, 8)
   log_cuantia  cuantia_rango_medio  cuantia_muy_alta  sector_competido  \
0    16.954344                    0                 0                 0   
1    17.655665                    0                 0                 1   
2    16.988133                    0                 0                 1   
3    15.424949                    0                 0                 1   
4    17.472383                    0                 0                 1   

   modalidad_abierta  entidad_alto_volumen  tiene_nit  tiene_municipio  
0                  0                     0          1                1  
1                  0                     0          1                1  
2                  0                     0          1                1  
3  

In [4]:
# Dividir en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Entrenamiento: {len(X_train):,} filas')
print(f'Prueba:        {len(X_test):,} filas')

Entrenamiento: 22,845 filas
Prueba:        5,712 filas


In [5]:
# Entrenar Random Forest
modelo = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_split=20,
    random_state=42,
    n_jobs=-1,
    class_weight='balanced',
)
modelo.fit(X_train, y_train)
print(' Modelo entrenado')

 Modelo entrenado


In [6]:
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report
import numpy as np
import pandas as pd

# =========================
# Evaluación del modelo
# =========================

y_pred = modelo.predict(X_test)

#  Verificar clases del modelo
print("Clases del modelo:", modelo.classes_)

# =========================
# Probabilidades (seguro)
# =========================
if hasattr(modelo, "predict_proba") and len(modelo.classes_) > 1:
    y_prob = modelo.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, y_prob)
else:
    print(" Modelo con una sola clase o sin predict_proba")
    y_prob = None
    auc = None

# =========================
# Distribución de datos
# =========================
print("\n Distribución y_train:")
print(y_train.value_counts())

print("\n Distribución y_test:")
print(y_test.value_counts())

# =========================
# Métricas
# =========================
print('\n MÉTRICAS DEL MODELO')
print('─' * 40)
print(f'Accuracy:  {accuracy_score(y_test, y_pred):.4f}')

if auc is not None:
    print(f'AUC-ROC:   {auc:.4f}')
else:
    print('AUC-ROC:   No disponible (una sola clase)')

# =========================
# Reporte de clasificación
# =========================
labels_presentes = np.unique(y_test)

print("\n Classification Report:")
if len(labels_presentes) == 2:
    print(classification_report(y_test, y_pred, target_names=['No ganó', 'Ganó']))
else:
    print(" Solo hay una clase en y_test")
    print(classification_report(y_test, y_pred))

# =========================
# Importancia de features
# =========================
if hasattr(modelo, "feature_importances_"):
    importancias = pd.Series(modelo.feature_importances_, index=X_train.columns)
    print('\n IMPORTANCIA DE FEATURES:')
    print(importancias.sort_values(ascending=False))
else:
    print("\n El modelo no soporta feature_importances_")

Clases del modelo: [0 1]

 Distribución y_train:
gano
1    15230
0     7615
Name: count, dtype: int64

 Distribución y_test:
gano
1    3808
0    1904
Name: count, dtype: int64

 MÉTRICAS DEL MODELO
────────────────────────────────────────
Accuracy:  0.4545
AUC-ROC:   0.3885

 Classification Report:
              precision    recall  f1-score   support

     No ganó       0.25      0.32      0.28      1904
        Ganó       0.61      0.52      0.56      3808

    accuracy                           0.45      5712
   macro avg       0.43      0.42      0.42      5712
weighted avg       0.49      0.45      0.47      5712


 IMPORTANCIA DE FEATURES:
log_cuantia             0.940161
sector_competido        0.017093
entidad_alto_volumen    0.015290
modalidad_abierta       0.013602
cuantia_rango_medio     0.007818
cuantia_muy_alta        0.006036
tiene_nit               0.000000
tiene_municipio         0.000000
dtype: float64


In [7]:
from pathlib import Path
import joblib

Path('models').mkdir(exist_ok=True)

joblib.dump(modelo, "../models/model.pkl")

print(' Modelo guardado en models/model.pkl')

 Modelo guardado en models/model.pkl
